# Finding the Optimal Hidden Layers and Neurons
GridSearchCV with **KerasClassifier** (churn) and **KerasRegressor** (salary).

Dataset: `Churn_Modelling_India_v2.csv`

Install once in your venv (also add `scikeras` to requirements.txt):
```
pip install scikeras
```

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# Old way (removed from Keras, gives ModuleNotFoundError):
# from keras.wrappers.scikit_learn import KerasClassifier
from scikeras.wrappers import KerasClassifier, KerasRegressor

## Load and prepare the data (same steps as before)

In [2]:
data = pd.read_csv('Churn_Modelling_India_v2.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(sparse_output=False)
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Delhi,Geography_Karnataka,Geography_Maharashtra
0,638,0,28,1,262577.80,2,0,1,540164.33,0,0.0,0.0,1.0
1,691,0,49,6,0.00,2,1,1,750959.23,0,0.0,1.0,0.0
2,701,1,74,6,172710.99,2,0,1,600400.82,0,0.0,0.0,1.0
3,681,0,41,7,250103.42,2,1,1,740038.64,0,0.0,0.0,1.0
4,704,1,40,5,373825.24,2,0,0,747950.61,0,0.0,0.0,1.0


---
# Part A: Classification (predict `Exited`) with KerasClassifier

In [3]:
X = data.drop('Exited', axis=1)
y = data['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### A function that builds any architecture
`neurons` = neurons in every hidden layer, `layers` = number of hidden layers.

In [4]:
def create_classification_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    # hidden layers
    for _ in range(layers):
        model.add(Dense(neurons, activation='relu'))
    # output layer: 1 neuron, sigmoid for binary classification
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

### Wrap the model for scikit-learn
Pass the defaults `neurons=32, layers=1`, otherwise GridSearchCV gives:
`ValueError: Invalid parameter neurons for estimator KerasClassifier`

In [ ]:
clf = KerasClassifier(model=create_classification_model,  # function that builds and compiles the Keras model
                      neurons=32,      # default number of neurons in each hidden layer
                      layers=1,        # default number of hidden layers
                      epochs=50,       # default number of full passes over the training data
                      batch_size=10,   # number of samples processed before weights are updated
                      verbose=0)       # 0 = no training logs printed (keeps the output clean)

# Values GridSearchCV will try (it overrides the defaults above)
param_grid = {
    'neurons': [16, 32, 64, 128],  # width of each hidden layer
    'layers': [1, 2],              # depth: how many hidden layers to stack
    'epochs': [50, 100]            # how long each candidate model trains
}
# 4 x 2 x 2 = 16 candidates, cv=3 -> 48 trainings (this takes time on a CPU)

In [ ]:
grid = GridSearchCV(estimator=clf, param_grid=param_grid,
                    n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)

print('Best accuracy:', grid_result.best_score_)
print('Best params  :', grid_result.best_params_)

In [ ]:
# All 16 candidates, best first
results = pd.DataFrame(grid_result.cv_results_)
results[['param_neurons', 'param_layers', 'param_epochs', 'mean_test_score']] \
    .sort_values('mean_test_score', ascending=False)

In [ ]:
# Check the best model on the test data
test_accuracy = grid_result.best_estimator_.score(X_test, y_test)
print('Test accuracy:', test_accuracy)

---
# Part B: Regression (predict `EstimatedSalary`) with KerasRegressor
Salary is a big number (around 7,00,000), so we predict it **in lakhs**: `7,40,000 -> 7.4`. Training is much faster and early results are better.

In [ ]:
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary'] / 100000      # salary in lakhs
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Same function, 3 changes for regression
1. Output `Dense(1)` with **no activation** (linear)
2. Loss `mean_absolute_error`
3. Metric `mae`

In [ ]:
def create_regression_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    for _ in range(layers):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1))                       # 1. linear output
    model.compile(optimizer='adam',
                  loss='mean_absolute_error',   # 2. regression loss
                  metrics=['mae'])              # 3. regression metric
    return model

### Wrap with KerasRegressor and search
`scoring='neg_mean_absolute_error'` makes GridSearchCV pick the lowest MAE. Without it, KerasRegressor ranks models by R².

scikit-learn always treats higher as better, so the score comes out **negative**: `-0.70` means MAE = 0.70 lakh = ₹70,000.

In [ ]:
reg = KerasRegressor(model=create_regression_model,
                     neurons=32, layers=1,
                     epochs=50, batch_size=10, verbose=0)

param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

grid = GridSearchCV(estimator=reg, param_grid=param_grid,
                    n_jobs=-1, cv=3, verbose=1,
                    scoring='neg_mean_absolute_error')
grid_result = grid.fit(X_train, y_train)

best_mae_lakh = -grid_result.best_score_
print('Best MAE     :', round(best_mae_lakh, 3), 'lakh =', round(best_mae_lakh * 100000), 'rupees')
print('Best params  :', grid_result.best_params_)

In [ ]:
results = pd.DataFrame(grid_result.cv_results_)
results['mae_rupees'] = -results['mean_test_score'] * 100000
results[['param_neurons', 'param_layers', 'param_epochs', 'mae_rupees']] \
    .sort_values('mae_rupees')

In [ ]:
# Check the best model on the test data, and compare with the baseline
from sklearn.metrics import mean_absolute_error

y_pred = grid_result.best_estimator_.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred) * 100000
baseline_mae = (y_test - y_train.mean()).abs().mean() *  100000

print(f'Test MAE    : ₹{test_mae:,.0f}')
print(f'Baseline MAE: ₹{baseline_mae:,.0f}  (always guessing the average)')

Next step: build the final ANN with the best `neurons` and `layers`, train it with EarlyStopping, and save it, exactly like before.